<a href="https://colab.research.google.com/github/RustyLegend/ML-Internship-Starter/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RustyLegend/ML-Internship-Starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [31]:
from google.colab import userdata
import duckdb

conn = duckdb.connect()

hf_key = userdata.get('HF_KEY')

conn.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{hf_key}')"
)

REL = 'hf://datasets/FlyRank/internship-warehouse'

TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_content_daily_performance': f"read_parquet('{REL}/fact_content_daily_performance')",
    'fact_content_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')"
}

query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(content_updated_date) AS non_null_updated_dates,
    COUNT(*) - COUNT(content_updated_date) AS missing_updated_dates,
    MIN(TRY_CAST(content_updated_date AS DATE)) AS earliest_update,
    MAX(TRY_CAST(content_updated_date AS DATE)) AS latest_update
FROM {TABLES['dim_content']}
"""

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## Baseline rule: Low CTR with meaningful search visibility

### Signal checks

**Signal 1 — Previous-30-day search visibility**

**Verdict: CONFIRMED**

Previous-30-day impressions show a clear positive relationship with subsequent visibility decline. Among pages with non-zero prior impressions, the decline rate increased from 52.45% in the 1–10 impression bucket to 76.99% in the 1,000+ bucket. The zero-impression bucket is not interpreted because a decline cannot be measured from a zero-impression starting point.

**Signal 2 — CTR relative to search position**

**Verdict: CONFIRMED**

CTR decreases systematically as search position becomes weaker, falling from 0.3604% for positions 1–3 to 0.1151% for positions 50+. This confirms that search position is important context when judging whether a page's CTR is unusually weak. Pages with invalid or zero position are excluded from this interpretation.

### Baseline rule

I prioritize pages that have meaningful search visibility but capture substantially fewer clicks than expected for their search-position tier.

A page receives the `REFRESH` action when:

- `impressions_prev30 >= 500`
- `0 < avg_position_prev30 <= 20`
- actual CTR is less than 50% of the expected CTR for its position tier.

The score is:

`score = impressions_prev30 × (1 - actual_ctr / expected_ctr)`

Every selected page receives one reason code:

`low_ctr_high_visibility`

Every selected page receives one action label:

`REFRESH`

This is a transparent hand-written baseline with no fitted model weights.

In [32]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Signal 1: Previous-30-day visibility audit

query = f"""
WITH content_level AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(impressions_prev30) AS impressions_prev30,
        SUM(impressions_last30) AS impressions_last30
    FROM {TABLES['fact_content_query_90d']}
    GROUP BY
        client_hash_id,
        content_hash_id
),

bucketed AS (
    SELECT
        *,
        CASE
            WHEN impressions_prev30 = 0 THEN '0'
            WHEN impressions_prev30 <= 10 THEN '1-10'
            WHEN impressions_prev30 <= 100 THEN '11-100'
            WHEN impressions_prev30 <= 1000 THEN '101-1000'
            ELSE '1000+'
        END AS volume_bucket
    FROM content_level
)

SELECT
    volume_bucket,
    COUNT(*) AS n,

    ROUND(
        100.0 * AVG(
            CASE
                WHEN impressions_prev30 > 0
                 AND impressions_last30 < impressions_prev30
                THEN 1.0
                ELSE 0.0
            END
        ),
        2
    ) AS decline_rate_pct

FROM bucketed

GROUP BY volume_bucket

ORDER BY
    CASE volume_bucket
        WHEN '0' THEN 1
        WHEN '1-10' THEN 2
        WHEN '11-100' THEN 3
        WHEN '101-1000' THEN 4
        WHEN '1000+' THEN 5
    END
"""

signal1_table = conn.sql(query).to_df()

print('Signal 1 table')
print(signal1_table)

# Signal 2: CTR versus search-position audit

query = f"""
WITH content_level AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(clicks_prev30) AS clicks_prev30,
        SUM(impressions_prev30) AS impressions_prev30,

        ANY_VALUE(avg_position_prev30) AS avg_position_prev30,
        SUM(impressions_last30) AS impressions_last30

    FROM {TABLES['fact_content_query_90d']}

    GROUP BY
        client_hash_id,
        content_hash_id
),

bucketed AS (
    SELECT
        *,
        CASE
            WHEN avg_position_prev30 > 0
                 AND avg_position_prev30 <= 3
                THEN '1-3'

            WHEN avg_position_prev30 > 3
                 AND avg_position_prev30 <= 10
                THEN '4-10'

            WHEN avg_position_prev30 > 10
                 AND avg_position_prev30 <= 20
                THEN '11-20'

            WHEN avg_position_prev30 > 20
                 AND avg_position_prev30 <= 50
                THEN '21-50'

            WHEN avg_position_prev30 > 50
                THEN '50+'

            ELSE 'No valid position'
        END AS position_bucket

    FROM content_level
)

SELECT
    position_bucket,
    COUNT(*) AS n,
    SUM(impressions_prev30) AS impressions,
    SUM(clicks_prev30) AS clicks,

    ROUND(
        100.0 * SUM(clicks_prev30)
        / NULLIF(SUM(impressions_prev30), 0),
        4
    ) AS ctr_pct,

    ROUND(
        100.0 * AVG(
            CASE
                WHEN impressions_prev30 > 0
                 AND impressions_last30 < impressions_prev30
                THEN 1.0
                ELSE 0.0
            END
        ),
        2
    ) AS decline_rate_pct

FROM bucketed

GROUP BY position_bucket

ORDER BY
    CASE position_bucket
        WHEN '1-3' THEN 1
        WHEN '4-10' THEN 2
        WHEN '11-20' THEN 3
        WHEN '21-50' THEN 4
        WHEN '50+' THEN 5
        ELSE 6
    END
"""

signal2_table = conn.sql(query).to_df()

print('\n Signal 2 table')
print(signal2_table)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Signal 1 table
  volume_bucket      n  decline_rate_pct
0             0  13666              0.00
1          1-10  19835             52.45
2        11-100  50503             66.62
3      101-1000  37373             71.93
4         1000+  12475             76.99


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


 Signal 2 table
     position_bucket      n  impressions   clicks  ctr_pct  decline_rate_pct
0                1-3   8467   10446726.0  37661.0   0.3605             61.72
1               4-10  30489   34286617.0  79637.0   0.2323             69.58
2              11-20  17159   11695332.0  21870.0   0.1870             72.42
3              21-50  25172    9094967.0  14860.0   0.1634             68.81
4                50+  34971    7393632.0   8501.0   0.1150             62.39
5  No valid position  17594    2309362.0   4400.0   0.1905             14.36


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv*

## Ranked baseline queue

The baseline selects pages with at least 500 previous-30-day impressions, an average search position between 1 and 20, and CTR below 50% of the expected CTR for their position tier.

The score combines visibility and CTR opportunity:

`score = impressions_prev30 × (1 - actual_ctr / expected_ctr)`

Higher scores represent pages with both meaningful search visibility and a larger CTR gap.

The queue uses one reason code, `low_ctr_high_visibility`, and one action label, `REFRESH`.

In [33]:
import os

os.makedirs("work/outputs", exist_ok=True)

query = f"""
WITH content_level AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(clicks_prev30) AS clicks_prev30,
        SUM(impressions_prev30) AS impressions_prev30,

        ANY_VALUE(avg_position_prev30) AS avg_position_prev30

    FROM {TABLES['fact_content_query_90d']}

    GROUP BY
        client_hash_id,
        content_hash_id
),

positioned AS (
    SELECT
        *,
        CASE
            WHEN avg_position_prev30 > 0
                 AND avg_position_prev30 <= 3
                THEN '1-3'

            WHEN avg_position_prev30 > 3
                 AND avg_position_prev30 <= 10
                THEN '4-10'

            WHEN avg_position_prev30 > 10
                 AND avg_position_prev30 <= 20
                THEN '11-20'

            WHEN avg_position_prev30 > 20
                 AND avg_position_prev30 <= 50
                THEN '21-50'

            WHEN avg_position_prev30 > 50
                THEN '50+'

            ELSE NULL
        END AS position_bucket

    FROM content_level
),

benchmarks AS (
    SELECT
        position_bucket,

        SUM(clicks_prev30)
        / NULLIF(SUM(impressions_prev30), 0)
        AS expected_ctr

    FROM positioned

    WHERE position_bucket IS NOT NULL
      AND impressions_prev30 > 0

    GROUP BY position_bucket
),

scored AS (
    SELECT
        p.client_hash_id,
        p.content_hash_id,

        p.impressions_prev30,
        p.clicks_prev30,
        p.avg_position_prev30,
        p.position_bucket,

        b.expected_ctr,

        p.clicks_prev30
        / NULLIF(p.impressions_prev30, 0)
        AS actual_ctr

    FROM positioned p

    JOIN benchmarks b
        ON p.position_bucket = b.position_bucket

    WHERE p.impressions_prev30 > 0
)

SELECT
    client_hash_id,
    content_hash_id,

    ROUND(impressions_prev30, 2) AS impressions_prev30,
    ROUND(clicks_prev30, 2) AS clicks_prev30,
    ROUND(avg_position_prev30, 2) AS avg_position_prev30,

    position_bucket,

    ROUND(100.0 * actual_ctr, 4) AS actual_ctr_pct,
    ROUND(100.0 * expected_ctr, 4) AS expected_ctr_pct,

    ROUND(
        100.0 * (1 - actual_ctr / expected_ctr),
        2
    ) AS ctr_gap_pct,

    ROUND(
        impressions_prev30
        * (1 - actual_ctr / expected_ctr),
        2
    ) AS score,

    'low_ctr_high_visibility' AS reason_code,

    'REFRESH' AS action

FROM scored

WHERE impressions_prev30 >= 500
  AND avg_position_prev30 > 0
  AND avg_position_prev30 <= 20
  AND actual_ctr < 0.50 * expected_ctr

ORDER BY score DESC
"""

baseline_queue = conn.sql(query).to_df()

output_path = "work/outputs/baseline_action_score.csv"

baseline_queue.to_csv(
    output_path,
    index=False
)

print(f"Rows in ranked queue: {len(baseline_queue):,}")
print(f"Written to: {output_path}")

print("\nTop 10:")
print(
    baseline_queue.head(10).to_string(index=False)
)

Rows in ranked queue: 6,202
Written to: work/outputs/baseline_action_score.csv

Top 10:
         client_hash_id          content_hash_id  impressions_prev30  clicks_prev30  avg_position_prev30 position_bucket  actual_ctr_pct  expected_ctr_pct  ctr_gap_pct     score             reason_code  action
client_8ddc46da5414ffd8 content_943dc881428182b8            313945.0          155.0                11.52           11-20          0.0494            0.1869        73.58 230993.79 low_ctr_high_visibility REFRESH
client_23a62021009f63c4 content_65c75874a23fca87            185602.0            5.0                 6.00            4-10          0.0027            0.2321        98.84 183447.39 low_ctr_high_visibility REFRESH
client_23a62021009f63c4 content_44f34c0a90047651            131836.0           12.0                 8.82            4-10          0.0091            0.2321        96.08 126664.95 low_ctr_high_visibility REFRESH
client_23a62021009f63c4 content_bddfdd871aa09fbe            110973.0    

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The top 20 pages from the baseline queue are reviewed manually. For each recommendation, I record the proposed action, the evidence that caused the page to rank highly, and what could make the recommendation wrong.

The baseline is intended to prioritize pages for human review rather than guarantee that a refresh will improve performance.

In [34]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Top-20 manual review

top20 = baseline_queue.head(20).copy()

print("TOP-20 REVIEW\n")

for rank, (_, row) in enumerate(top20.iterrows(), start=1):

    print(
        f"{rank}. "
        f"ACTION: {row['action']} | "
        f"WHY: {row['impressions_prev30']:,.0f} previous-30d impressions, "
        f"position {row['avg_position_prev30']:.1f}, "
        f"CTR {row['actual_ctr_pct']:.4f}% vs "
        f"{row['expected_ctr_pct']:.4f}% expected "
        f"({row['ctr_gap_pct']:.2f}% below expected). | "
        f"WRONG IF: the low CTR is explained by factors not captured "
        f"by search position and visibility, or the page does not have "
        f"a meaningful content or metadata opportunity."
    )

TOP-20 REVIEW

1. ACTION: REFRESH | WHY: 313,945 previous-30d impressions, position 11.5, CTR 0.0494% vs 0.1869% expected (73.58% below expected). | WRONG IF: the low CTR is explained by factors not captured by search position and visibility, or the page does not have a meaningful content or metadata opportunity.
2. ACTION: REFRESH | WHY: 185,602 previous-30d impressions, position 6.0, CTR 0.0027% vs 0.2321% expected (98.84% below expected). | WRONG IF: the low CTR is explained by factors not captured by search position and visibility, or the page does not have a meaningful content or metadata opportunity.
3. ACTION: REFRESH | WHY: 131,836 previous-30d impressions, position 8.8, CTR 0.0091% vs 0.2321% expected (96.08% below expected). | WRONG IF: the low CTR is explained by factors not captured by search position and visibility, or the page does not have a meaningful content or metadata opportunity.
4. ACTION: REFRESH | WHY: 110,973 previous-30d impressions, position 1.0, CTR 0.0009% v

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*


The baseline is intentionally simple, so some candidates near the decision boundaries may be weak recommendations. I inspect edge cases to identify where the rule could fail.

A weak pick is not necessarily a wrong recommendation; it is a page where the evidence is less convincing and human review is especially important.

In [35]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Find candidates closest to the rule's decision boundaries.
# These are useful weak-pick examples for manual review.

weak_picks = baseline_queue.copy()

weak_picks["volume_distance"] = (
    weak_picks["impressions_prev30"] - 500
)

weak_picks["position_distance"] = (
    20 - weak_picks["avg_position_prev30"]
)

weak_picks["ctr_threshold_ratio"] = (
    weak_picks["actual_ctr_pct"]
    / weak_picks["expected_ctr_pct"]
)

# Pages closest to the CTR threshold are the least decisive.
weak_picks["ctr_distance_from_threshold"] = abs(
    weak_picks["ctr_threshold_ratio"] - 0.50
)

weak_picks = weak_picks.sort_values(
    "ctr_distance_from_threshold"
).head(5)

print("WEAK PICKS\n")

print(
    weak_picks[
        [
            "client_hash_id",
            "content_hash_id",
            "impressions_prev30",
            "avg_position_prev30",
            "actual_ctr_pct",
            "expected_ctr_pct",
            "ctr_gap_pct",
            "score",
            "reason_code",
            "action"
        ]
    ].to_string(index=False)
)

WEAK PICKS

         client_hash_id          content_hash_id  impressions_prev30  avg_position_prev30  actual_ctr_pct  expected_ctr_pct  ctr_gap_pct   score             reason_code  action
client_73cda7b4e4f265ea content_9490fac33e6f6794              2586.0                 9.59          0.1160            0.2321        50.01 1293.24 low_ctr_high_visibility REFRESH
client_23a62021009f63c4 content_80d02d2d5eaa18e9              1724.0                 7.48          0.1160            0.2321        50.01  862.16 low_ctr_high_visibility REFRESH
client_73cda7b4e4f265ea content_688547d0e538c247              3213.0                12.05          0.0934            0.1869        50.03 1607.49 low_ctr_high_visibility REFRESH
client_73cda7b4e4f265ea content_6cdfbd209101f748              3212.0                10.43          0.0934            0.1869        50.02 1606.49 low_ctr_high_visibility REFRESH
client_62f4a7e64f5e0096 content_80b1d2b17cab1137              1071.0                10.79          0.09

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.